# OLS Regression Analysis — Qinghe Park Survey (N=232)

**Dataset**: `combined_clean_232.xlsx`  
**Research question**: Do participation level and public awareness (PAW) predict ecological satisfaction outcomes, controlling for demographics?

---

## Model structure (per outcome)

| Model | Predictors |
|---|---|
| **M1** | Demographics only (controls) |
| **M2** | Demographics + Info provision + Consultation |
| **M3** | Demographics + Info provision + Consultation + PAW |

## Regression equations

**M1:**
$$\hat{Y} = b_0 + b_1\,\text{Wave2} + b_2\,\text{Gender} + b_3\,\text{Age} + b_4\,\text{Edu} + b_5\,\text{Distance} + b_6\,\text{VisitFreq} + \varepsilon$$

**M2:**
$$\hat{Y} = \underbrace{b_0 + \ldots + b_6}_{\text{demographics}} + b_7\,\text{InfoProvision} + b_8\,\text{Consultation} + \varepsilon$$

**M3 (full model):**
$$\hat{Y} = \underbrace{b_0 + \ldots + b_6}_{\text{demographics}} + \underbrace{b_7\,\text{InfoProvision} + b_8\,\text{Consultation}}_{\text{participation}} + \underbrace{b_9\,\text{PAW}}_{\text{awareness}} + \varepsilon$$

**Standardised coefficient:**
$$\beta^* = b_k \times \frac{\sigma_{x_k}}{\sigma_Y}$$

**Delta R²:**
$$\Delta R^2 = R^2_{M3} - R^2_{M1}$$

---

## Variable coding

| Variable | Column | Values | Rationale |
|---|---|---|---|
| Survey wave | `wave2` | 1 = Wave 2, 0 = Wave 1 (ref) | Controls for time effects |
| Gender | `gender_bin` | 1 = Female, 0 = Male (ref) | Binary |
| Age group | `age_num` | 1 = <25 … 4 = >45 | Ordinal |
| Education | `edu_num` | 1 = High school … 4 = Master+ | Ordinal |
| Distance to park | `dist_num` | 1 = <1 km … 5 = >40 km | Ordinal |
| Visit frequency | `freq_num` | 1 = Rarely … 5 = Daily | Ordinal |
| Info provision | `Info_Provision_score` | 0–1 continuous | Proportion of info channels used |
| Consultation | `Consultation_score` | 0–1 continuous | Proportion of consultation channels used |
| Public awareness | `PAW` | 0–1 continuous | Mean of signage + terminology items |

---
## Block 1 — Setup & variable coding

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

DATA = 'combined_clean_232.xlsx'
OUT  = 'regression_results.xlsx'

df = pd.read_excel(DATA, sheet_name='Data')

# Wave dummy: 1 = Wave 2, 0 = Wave 1 (reference)
df['wave2'] = (df['wave'] == 'wave2').astype(float)

# All other demographics already coded as ordinal integers in this dataset:
#   gender_bin  : 0 = Male, 1 = Female
#   age_num     : 1 (Under 25) to 4 (Over 45)
#   edu_num     : 1 (High school) to 4 (Master+)
#   dist_num    : 1 (<1 km) to 5 (>40 km)
#   freq_num    : 1 (Rarely) to 5 (Daily)

DEMO_COLS = ['wave2', 'gender_bin', 'age_num', 'edu_num', 'dist_num', 'freq_num']
PART_COLS = ['Info_Provision_score', 'Consultation_score']
PAW_COLS  = ['PAW']

OUTCOMES = [
    ('EMQ',     'EMQ'),
    ('EHB',     'EHB'),
    ('ECO',     'ECO'),
    ('MGT',     'MGT'),
    ('OVERALL', 'OVERALL'),
]

print(f'Total N = {len(df)}')
print(f'Wave 1: {(df["wave"]=="wave1").sum()}  |  Wave 2: {(df["wave"]=="wave2").sum()}')
print(f'Predictors — M1: {len(DEMO_COLS)}  |  M3: {len(DEMO_COLS)+len(PART_COLS)+len(PAW_COLS)}')
print()
print('NaN counts in predictors:')
for c in DEMO_COLS + PART_COLS + PAW_COLS:
    print(f'  {c}: {df[c].isna().sum()}')
print('Setup complete.')

---
## Block 2 — Run OLS regressions (M1 / M2 / M3)

Three nested models per outcome.

> **Important — common estimation sample.** All three models for a given outcome are
> fit on the *same* complete-case sample: the rows with no missing value on the
> outcome or on **any** M3 predictor. This is required for the models to be genuinely
> nested. If each model were instead given its own listwise sample, M1 and M3 would be
> estimated on different respondents and $\Delta R^2$ would be meaningless — it could
> even come out negative, which is impossible for correctly nested models.

### Step-by-step walkthrough

**Step 1 — Common complete-case sample**: drop rows missing the outcome or any M3
predictor. `n` is then constant across M1/M2/M3 within each outcome.

**Step 2 — Add constant**: `sm.add_constant(X)` prepends a column of ones so OLS estimates the intercept $b_0$.

**Step 3 — Fit OLS**: solves $\hat{\mathbf{b}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$ — minimises the sum of squared residuals.

**Step 4 — Standardised betas**: $\beta^*_k = b_k \times \frac{SD(x_k)}{SD(Y)}$, computed on the common sample.

**Step 5 — Significance**: t-test on each coefficient ($H_0: b_k = 0$), coded \*\*\* / \*\* / \* / † / ns.

**Step 6 — $\Delta R^2$ and the F-change test**: the increment from adding a block of
$q$ predictors is tested with

$$F_{\text{change}} = \frac{\left(R^2_{\text{full}} - R^2_{\text{reduced}}\right) / q}
{\left(1 - R^2_{\text{full}}\right) / \left(n - k_{\text{full}} - 1\right)}$$

on $(q,\; n - k_{\text{full}} - 1)$ degrees of freedom. This tests whether the **added
block** explains variance — which is a different question from the overall model $F$.


In [ ]:
from scipy import stats

FULL_COLS = DEMO_COLS + PART_COLS + PAW_COLS

def run_ols(y_col, x_cols, sample):
    X = sm.add_constant(sample[x_cols])
    y = sample[y_col]
    return sm.OLS(y, X).fit()

def std_betas(result, sample, x_cols, y_col):
    sy = sample[y_col].std(ddof=1)
    out = {}
    for col in x_cols:
        sx = sample[col].std(ddof=1)
        b  = result.params.get(col, np.nan)
        out[col] = (b * sx / sy) if (sy > 0 and sx > 0) else np.nan
    return out

def f_change(r2_full, r2_red, k_full, k_red, n):
    """F-test for the R2 increment between nested models on the SAME sample."""
    q   = k_full - k_red
    dfd = n - k_full - 1
    dr2 = r2_full - r2_red
    f   = (dr2 / q) / ((1 - r2_full) / dfd)
    return dr2, f, stats.f.sf(f, q, dfd)

all_rows = []

for out_col, out_lbl in OUTCOMES:
    # Common complete-case sample (= the M3 listwise sample).
    # All three models use this SAME sample, so they are genuinely nested
    # and dR2 / F-change are valid.
    sample = df[[out_col] + FULL_COLS].dropna()
    n_used = len(sample)

    models = {'M1': DEMO_COLS,
              'M2': DEMO_COLS + PART_COLS,
              'M3': FULL_COLS}

    prev_r2 = prev_k = None
    for m_name, x_cols in models.items():
        res = run_ols(out_col, x_cols, sample)
        sb  = std_betas(res, sample, x_cols, out_col)
        r2  = res.rsquared
        k   = len(x_cols)

        if prev_r2 is None:
            dr2 = f_ch = p_ch = np.nan
        else:
            dr2, f_ch, p_ch = f_change(r2, prev_r2, k, prev_k, n_used)
        prev_r2, prev_k = r2, k

        for col in x_cols:
            b   = res.params.get(col, np.nan)
            se  = res.bse.get(col, np.nan)
            t   = res.tvalues.get(col, np.nan)
            p   = res.pvalues.get(col, np.nan)
            ci  = res.conf_int().loc[col]
            sig = ('***' if p < 0.001 else ('**' if p < 0.01 else
                   ('*'  if p < 0.05  else ('t'  if p < 0.10  else 'ns'))))
            all_rows.append({
                'Outcome': out_lbl, 'Model': m_name, 'Predictor': col,
                'n': n_used,
                'B': round(b, 4), 'B_std': round(sb.get(col, np.nan), 4),
                'SE': round(se, 4), 't': round(t, 3), 'p': round(p, 4),
                'CI_lo': round(ci[0], 4), 'CI_hi': round(ci[1], 4),
                'Sig': sig,
                'R2': round(r2, 4),
                'dR2':      '' if np.isnan(dr2)  else round(dr2, 4),
                'F_change': '' if np.isnan(f_ch) else round(f_ch, 3),
                'dR2_p':    '' if np.isnan(p_ch) else round(p_ch, 4),
                'F': round(res.fvalue, 3), 'F_p': round(res.f_pvalue, 4),
            })

df_reg = pd.DataFrame(all_rows)

# Sanity check: n must now be identical across M1/M2/M3 within each outcome.
assert (df_reg.groupby('Outcome')['n'].nunique() == 1).all(), \
    'n differs across models - nesting is broken'

print(f'Regression complete: {len(df_reg)} rows across {len(OUTCOMES)*3} models')
print('\nCommon estimation sample per outcome (same for M1/M2/M3):')
print(df_reg.groupby('Outcome')['n'].first().to_string())


---
## Block 3 — Summary tables

### 3a — Model fit

In [ ]:
fit_cols = ['Outcome', 'Model', 'n', 'R2', 'dR2', 'F_change', 'dR2_p', 'F', 'F_p']
df_fit   = (df_reg[fit_cols]
            .drop_duplicates(subset=['Outcome', 'Model'])
            .reset_index(drop=True))
display(df_fit)


### 3b — Full M3 coefficients per outcome

In [ ]:
for out_lbl in [o[1] for o in OUTCOMES]:
    sub = df_reg[(df_reg['Outcome'] == out_lbl) & (df_reg['Model'] == 'M3')].copy()
    print(f'\n── {out_lbl}  (n={sub["n"].iloc[0]}, R²={sub["R2"].iloc[0]:.3f}) ──')
    display(sub[['Predictor', 'B', 'B_std', 'SE', 't', 'p', 'Sig', 'CI_lo', 'CI_hi']]
            .reset_index(drop=True))

### 3c — Key predictors across all outcomes (M3)

In [ ]:
key_preds = ['wave2', 'gender_bin', 'age_num', 'edu_num', 'dist_num', 'freq_num',
             'Info_Provision_score', 'Consultation_score', 'PAW']
df_key = (df_reg[
    (df_reg['Model'] == 'M3') & (df_reg['Predictor'].isin(key_preds))
][['Outcome', 'Predictor', 'n', 'B', 'B_std', 'SE', 't', 'p', 'Sig', 'R2']]
 .reset_index(drop=True))
display(df_key)

---
## Block 4 — Export to Excel

In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

H_FILL   = PatternFill('solid', fgColor='1F4E79')
H_FONT   = Font(bold=True, color='FFFFFF')
SIG_FILL = PatternFill('solid', fgColor='E2EFDA')
NS_FILL  = PatternFill('solid', fgColor='F2F2F2')

def style_ws(ws, sig_col=None):
    headers = [c.value for c in ws[1]]
    for cell in ws[1]:
        cell.fill = H_FILL; cell.font = H_FONT
        cell.alignment = Alignment(horizontal='center')
    if sig_col and sig_col in headers:
        si = headers.index(sig_col) + 1
        for row in ws.iter_rows(min_row=2):
            val = str(row[si-1].value)
            row[si-1].fill = SIG_FILL if val not in ('ns', 'None', '') else NS_FILL
    for col in ws.columns:
        w = max(len(str(c.value or '')) for c in col) + 3
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w, 28)

with pd.ExcelWriter(OUT, engine='openpyxl') as writer:
    df_fit.to_excel(writer, sheet_name='Model_fit',      index=False)
    df_key.to_excel(writer, sheet_name='Key_predictors', index=False)
    df_reg.to_excel(writer, sheet_name='Full_results',   index=False)
    wb = writer.book
    for sh, sc in [('Model_fit', None), ('Key_predictors', 'Sig'), ('Full_results', 'Sig')]:
        style_ws(wb[sh], sig_col=sc)

print(f'Saved: {OUT}')

---
## Block 5 — Publication-quality regression table figure

Renders the M3 results as a booktabs-style matplotlib figure (PNG 300 dpi + PDF).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib import rcParams
rcParams['font.family'] = 'serif'
rcParams['font.serif']  = ['Times New Roman', 'DejaVu Serif', 'Georgia']

OUTCOMES_FIG = ['EMQ', 'EHB', 'ECO', 'MGT', 'OVERALL']

PRED_CONFIG = [
    ('wave2',                'Demographics',     'Survey wave (ref: Wave 1)',        False),
    ('gender_bin',           None,               'Gender (ref: Male)',               False),
    ('age_num',              None,               'Age group (ordinal 1-4)',          False),
    ('edu_num',              None,               'Education (ordinal 1-4)',          False),
    ('dist_num',             None,               'Distance to park (ordinal 1-5)',   False),
    ('freq_num',             None,               'Visit frequency (ordinal 1-5)',    False),
    ('Info_Provision_score', 'Participation',    'Information provision score',      False),
    ('Consultation_score',   None,               'Consultation score',               False),
    ('PAW',                  'Public Awareness', 'PAW composite',                    False),
]

def _stars(sig):
    return {'***':'***','**':'**','*':'*','t':'†','ns':''}[sig]

def _fmt_beta(b_std, sig):
    if b_std is None or (isinstance(b_std, float) and np.isnan(b_std)): return '-'
    sign   = '-' if b_std < 0 else ' '
    digits = f'{round(abs(b_std)*1000):03d}'
    return f'{sign}.{digits}{_stars(sig)}'

def _fmt_bse(b, se):
    if b is None or (isinstance(b, float) and np.isnan(b)): return '-'
    return f"{'−' if b<0 else ''}{abs(b):.3f} ({se:.3f})"

def _style(sig):
    if sig in ('***','**','*'): return 'black', True
    if sig == 't':              return '#555555', False
    return '#999999', False

def _dr2_str(out, fm3, fm1):
    # M1 -> M3 increment, tested with the F-change test (not the overall model F)
    dr2, _f, p = f_change(fm3.loc[out,'R2'], fm1.loc[out,'R2'],
                          len(FULL_COLS), len(DEMO_COLS), fm3.loc[out,'n'])
    star = '***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else ('†' if p<.10 else '')))
    return f'{dr2:.3f}{star}'

def _f_str(out, fm3):
    fp   = fm3.loc[out,'F_p']
    star = '***' if fp<.001 else ('**' if fp<.01 else ('*' if fp<.05 else ('†' if fp<.10 else '')))
    return f'{fm3.loc[out,"F"]:.2f}{star}'

fm3 = df_fit[df_fit['Model']=='M3'].set_index('Outcome')
fm1 = df_fit[df_fit['Model']=='M1'].set_index('Outcome')
m3  = df_reg[df_reg['Model']=='M3'].set_index(['Outcome','Predictor'])

def lookup(out, col):
    key = (out, col)
    if key not in m3.index: return None, None, None, 'ns'
    r = m3.loc[key]
    return (float(r['B_std']) if not pd.isna(r['B_std']) else None,
            float(r['B'])     if not pd.isna(r['B'])     else None,
            float(r['SE'])    if not pd.isna(r['SE'])    else None,
            str(r['Sig']))

# Build table rows
rows = []; cur_sec = None
for col, section, label, _ in PRED_CONFIG:
    if section and section != cur_sec:
        rows.append(('section', section)); cur_sec = section
    cells = []
    for o in OUTCOMES_FIG:
        b_std, b, se, sig = lookup(o, col)
        cells.append((_fmt_beta(b_std, sig), _fmt_bse(b, se), *_style(sig)))
    rows.append(('data', label, cells))
rows.append(('rule',))
for lbl, vals in [
    ('n',            [str(int(fm3.loc[o,'n'])) for o in OUTCOMES_FIG]),
    ('R²',           [f'{fm3.loc[o,"R2"]:.3f}' for o in OUTCOMES_FIG]),
    ('ΔR² (M1→M3)', [_dr2_str(o, fm3, fm1) for o in OUTCOMES_FIG]),
    ('F (model)',    [_f_str(o, fm3) for o in OUTCOMES_FIG]),
]:
    rows.append(('stat', lbl, vals))

# Draw figure
N = len(OUTCOMES_FIG)
ROW_H=0.28; SEC_H=0.30; RULE_H=0.06; HEAD_H=0.50; FOOT_H=1.0; TOP_H=0.55
FIG_W=14.0
FIG_H = max(HEAD_H + sum(SEC_H if r[0]=='section' else
                         RULE_H if r[0]=='rule' else ROW_H for r in rows)
            + FOOT_H + TOP_H + 0.4, 7.0)
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

L=0.03; R=0.98; PF=0.22; DF=(1-PF)/(N*2)
def cx(c): return L if c==0 else L+PF*(R-L)+(c-1)*DF*(R-L)
def hl(y, lw, x0=None, x1=None):
    fig.add_artist(mlines.Line2D([x0 or L, x1 or R],[y,y],
        transform=fig.transFigure,color='#1c1c1c',linewidth=lw,clip_on=False))

y = 0.97
fig.text(L, y, 'Table 1   OLS Regression of Ecological Satisfaction on '
         'Participation and Public Awareness (N=232)',
         transform=fig.transFigure, fontsize=10, fontweight='bold', va='top', ha='left')
y -= TOP_H*0.55/FIG_H
fig.text(L, y, 'Full model (M3); β* = standardised, B (SE) = unstandardised. '
         'Controlling for demographics.',
         transform=fig.transFigure, fontsize=8.5, style='italic', color='#444444', va='top', ha='left')
y -= TOP_H*0.45/FIG_H
hl(y, 1.5)
y -= HEAD_H*0.46/FIG_H
for i, out in enumerate(OUTCOMES_FIG):
    mid = (cx(1+i*2)+cx(3+i*2))/2
    fig.text(mid, y, out, transform=fig.transFigure,
             fontsize=9, fontweight='bold', ha='center', va='center')
    hl(y-0.012, 0.5, cx(1+i*2)+0.002, cx(3+i*2)-0.002)
y -= HEAD_H*0.54/FIG_H
fig.text(L+0.002, y, 'Predictor', transform=fig.transFigure,
         fontsize=8.5, style='italic', va='center', ha='left')
for i in range(N):
    fig.text((cx(1+i*2)+cx(2+i*2))/2, y, 'β*',
             transform=fig.transFigure, fontsize=8.5, ha='center', va='center')
    fig.text((cx(2+i*2)+cx(3+i*2))/2, y, 'B (SE)',
             transform=fig.transFigure, fontsize=8.5, ha='center', va='center')
y -= 0.014; hl(y, 1.0)

for row in rows:
    k = row[0]
    if k == 'section':
        y -= SEC_H/FIG_H*0.35
        hl(y+SEC_H/FIG_H*0.35-0.003, 0.4)
        fig.text(L+0.003, y, row[1], transform=fig.transFigure,
                 fontsize=8.5, style='italic', color='#444444', va='center', ha='left')
        y -= SEC_H/FIG_H*0.65
    elif k == 'data':
        _, label, cells = row
        fig.text(L+0.018, y, label, transform=fig.transFigure,
                 fontsize=8.5, va='center', ha='left')
        for i, (bs, bse, color, bold) in enumerate(cells):
            kw = dict(transform=fig.transFigure, fontsize=8, ha='center',
                      va='center', color=color, fontfamily='monospace',
                      fontweight='bold' if bold else 'normal')
            fig.text((cx(1+i*2)+cx(2+i*2))/2, y, bs, **kw)
            fig.text((cx(2+i*2)+cx(3+i*2))/2, y, bse, **{**kw,'fontsize':7.5})
        y -= ROW_H/FIG_H
    elif k == 'rule':
        hl(y+ROW_H/FIG_H*0.3, 0.8); y -= RULE_H/FIG_H
    elif k == 'stat':
        _, label, vals = row
        fig.text(L+0.003, y, label, transform=fig.transFigure,
                 fontsize=8.5, style='italic', color='#444444', va='center', ha='left')
        for i, v in enumerate(vals):
            fig.text((cx(1+i*2)+cx(3+i*2))/2, y, v,
                     transform=fig.transFigure, fontsize=8, ha='center', va='center')
        y -= ROW_H/FIG_H

hl(y+ROW_H/FIG_H*0.3, 1.5)
y -= 0.018
for line in [
    'Note. EMQ = Environmental and ecological quality; EHB = Environmental hydrological benefits; '
    'ECO = Biodiversity satisfaction; MGT = Park management satisfaction; OVERALL = mean of all four constructs.',
    'β* = standardised coefficient; B = unstandardised; SE = standard error. '
    'All demographic variables pre-coded as ordinal integers.',
    'ΔR² = increment from M1 (demographics only) to M3 (full model), tested by F-change. '
    'For each outcome all three models are fit on a common complete-case sample.',
    '† p < .10   * p < .05   ** p < .01   *** p < .001',
]:
    fig.text(L, y, line, transform=fig.transFigure,
             fontsize=7.5, color='#555555', va='top', ha='left')
    y -= 0.030

plt.savefig('regression_table.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('regression_table.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: regression_table.png / .pdf')